In [ ]:
# # 九曜：天墟 EP01 — 配音試作：BreezyVoice（台灣國語，Kaggle）
#
# Qwen3-TTS 用文字設計的聲音一直偏粵語腔（v2、v3）。改試 BreezyVoice（聯發科研究院，專為台灣國語訓練，Apache 2.0）：
# 拿 v3 挑出的 旁白、蕭曜霖 c2 參考聲音（已內嵌在本筆記本）讓 BreezyVoice 唸台詞，看能不能保留音色、口音變台灣腔。
# 先只做這兩個角色；每句 3 次。「九曜」的「曜」用注音 `[:ㄧㄠ4]` 指定讀音。
#
# Kaggle：GPU T4 x2、Internet On → Save Version → Save & Run All。
# BreezyVoice 需要 Python 3.10，第一格會另外裝一個 3.10 環境（放在 /tmp，不會進 Output），約 10～15 分鐘。
# 跑完：右側 Output → `/kaggle/working` → EP01_voice_breezy_tiny.zip（⋮ → Download）傳給 Claude。
# 第二版（2026-09-29）：第一次跑出空檔，看不到原因 → 這版把安裝／執行紀錄（`log_*.txt`）一起打包；就算又失敗，也請照樣下載 zip 傳給 Claude。


In [ ]:
# 安裝：Python 3.10 虛擬環境；每一步的輸出都寫進 LOG（會一起打包給 Claude 看）
import subprocess, os, shutil, shlex
LOGDIR = "/kaggle/working/voice_breezy_tiny"; os.makedirs(LOGDIR, exist_ok=True)
def sh(cmd, log, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True, env={**os.environ, "PYTHONUTF8": "1"})
    with open(f"{LOGDIR}/{log}", "a", encoding="utf-8") as f: f.write(f"\n$ {cmd}\n[exit {r.returncode}]\n{r.stdout[-20000:]}\n{r.stderr[-20000:]}\n")
    print(f"[exit {r.returncode}] {cmd[:100]}"); return r.returncode
sh("pip install -q uv", "log_install.txt")
sh("uv python install 3.10", "log_install.txt")
shutil.rmtree("/tmp/BreezyVoice", ignore_errors=True)
sh("git clone -q https://github.com/mtkresearch/BreezyVoice.git /tmp/BreezyVoice", "log_install.txt")
sh("uv venv -q -p 3.10 /tmp/bv", "log_install.txt")
# 拿掉：網頁介面／訓練加速（推論用不到）、ttsfrd（modelscope 下載常失敗，程式會改用 WeTextProcessing）
keep = [l.strip() for l in open("/tmp/BreezyVoice/requirements.txt") if l.strip() and not l.startswith("#")
        and not any(l.startswith(x) for x in ("gradio", "fastapi", "deepspeed", "tensorboard", "https://www.modelscope"))]
open("/tmp/req_kaggle.txt", "w").write("\n".join(keep))
PIP = "uv pip install -p /tmp/bv/bin/python --index-strategy unsafe-best-match --extra-index-url https://download.pytorch.org/whl/cu118"
# openai-whisper 舊版打包要 pkg_resources（新版 setuptools 已移除）→ 先裝舊 setuptools，whisper 關掉 build isolation
sh(f"{PIP} 'setuptools<70' wheel", "log_install.txt")
PIP += " --no-build-isolation-package openai-whisper"
if sh(f"{PIP} -r /tmp/req_kaggle.txt", "log_install.txt") != 0:
    # 整包失敗就一個一個裝，記下哪個裝不起來
    for pkg in keep:
        if not pkg.startswith("--"): sh(f"{PIP} {shlex.quote(pkg)}", "log_install.txt")
sh("/tmp/bv/bin/python -c \"import torch; import whisper; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'whisper ok')\"", "log_install.txt")


In [ ]:
# 參考聲音（v3 c2，16kHz WAV 內嵌）與台詞表
import base64, csv, json, os
REFS = json.loads(r'''{"VO_c2": {"text": "\u7fa4\u5c71\u4e4b\u4e0a\uff0c\u96f2\u6d77\u7ffb\u6e67\uff0c\u4e00\u5ea7\u53e4\u8001\u7684\u5b78\u9662\uff0c\u975c\u975c\u77d7\u7acb\u5728\u90a3\u88e1\u3002", "b64": "..."}, "C05_c2": {"text": "\u6240\u6709\u4eba\u807d\u597d\uff0c\u660e\u5929\u4e00\u65e9\u5230\u6f14\u6b66\u5834\u96c6\u5408\uff0c\u8ab0\u90fd\u4e0d\u51c6\u9072\u5230\u3002", "b64": "..."}}''')
ROWS = [["VO_c2", "E01-01_c2_1_旁白", "九曜[:ㄧㄠ4]界，天玄院。"], ["C05_c2", "E01-07_c1_1_蕭曜霖", "不用等明天。"], ["C05_c2", "E01-07_c4_1_蕭曜霖", "半刻鐘後，新生演武場集合。"]]
TAKES = 3
os.makedirs("/tmp/bv_ref", exist_ok=True); OUT = "/kaggle/working/voice_breezy"; os.makedirs(OUT, exist_ok=True)
for k, v in REFS.items(): open(f"/tmp/bv_ref/{k}.wav", "wb").write(base64.b64decode(v["b64"]))
with open("/tmp/bv_input.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["speaker_prompt_audio_filename", "speaker_prompt_text_transcription", "content_to_synthesize", "output_audio_filename"])
    for ref, name, text in ROWS:
        for t in range(1, TAKES + 1): w.writerow([ref, REFS[ref]["text"], text, f"{name}_t{t}"])
print(open("/tmp/bv_input.csv", encoding="utf-8").read())


In [ ]:
# 生成（第一次會自動下載模型）；輸出寫進 log_run.txt
sh("/tmp/bv/bin/python batch_inference.py --csv_file /tmp/bv_input.csv --speaker_prompt_audio_folder /tmp/bv_ref --output_audio_folder /kaggle/working/voice_breezy", "log_run.txt", cwd="/tmp/BreezyVoice")
sh("find /kaggle/working/voice_breezy /tmp/BreezyVoice/results -type f 2>/dev/null | head -50", "log_run.txt")
print(open(f"{LOGDIR}/log_run.txt", encoding="utf-8").read()[-3000:])


In [ ]:
# 打包上傳用小檔（MP3＋安裝／執行紀錄）；就算失敗也會打包紀錄
import glob
dst = LOGDIR
for w in glob.glob(f"{OUT}/**/*.wav", recursive=True) + glob.glob("/tmp/BreezyVoice/results/**/*.wav", recursive=True):
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", w, "-t", "12", "-ac", "1", "-b:a", "48k", f"{dst}/{os.path.basename(w)[:-4]}.mp3"])
shutil.make_archive("/kaggle/working/EP01_voice_breezy_tiny", "zip", dst)
print(len(glob.glob(f"{dst}/*.mp3")), "個聲音檔 →", f"EP01_voice_breezy_tiny.zip {os.path.getsize('/kaggle/working/EP01_voice_breezy_tiny.zip')/1e6:.1f} MB（含 log_*.txt）")
